# 2. Fine-tune MMELON

Fine-tunes MMELON on every fold, picks the epoch by the fold average, trains a final model on all training
rows for that many epochs, and scores the test set. This is the `finetune` method, exactly what
`run.py --method finetune` does; the code is in [../methods/finetune.py](../methods/finetune.py) and this
notebook only chooses the options and looks at the results.

Replaces `MMELON_finetuning_cv`, `MMELON_singleview_finetuning_cv`, `MMELON_finetuning`,
`MMELON_singleview_finetuning` and the four `*_inference*` notebooks in
[../../notebooks/](../../notebooks/).

**What it does, in order**
1. Writes each fold's training and test rows, then trains all folds side by side, one epoch at a time.
2. After each epoch, averages the metrics over the folds and keeps the best epoch by `SELECT_METRICS`
   (`hit@200`, then `hit@100`, `hit@50`, `rocauc`); stops after `PATIENCE` epochs without improvement.
3. Scores each fold's test rows with that fold's model (splits `fold0`, `fold1`, …).
4. Scores the test file with the fold models averaged (`test_ensemble`).
5. Trains one final model on all training rows for the selected number of epochs and scores the test file
   with it (`test`). The test file never takes part in choosing anything.

**This needs a GPU.** On a CPU-only machine it takes hours; try `EPOCHS = 1` on a small dataset first.
Multi-view also needs memory: set `NUM_WORKERS = 0` or `2` if the process gets killed.

## Settings

In [ ]:
EXPERIMENT = "sample-benchmark"
RUN_NAME = "finetune_text"                      # folder under the experiment; must not hold a run yet
VIEW = "text"                                   # "multiview", "text", "image" or "graph"
TEST_PATH = "../../sample_data/test.parquet"

EPOCHS, PATIENCE = 50, 5                        # maximum epochs, and when to give up
BATCH_SIZE, LEARNING_RATE = 16, 2e-5
BALANCE_CLASSES = True                          # weight the loss by the class counts
FINAL = True                                    # also train the final model on all training rows
NUM_WORKERS = None                              # None = all CPUs (up to 24); 0-2 on a small machine

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Make the benchmark code importable (works from this folder or from the repo root)
BENCHMARK = next(p / rel for p in [Path.cwd(), *Path.cwd().parents] for rel in ("", "aircheck/benchmark")
                 if (p / rel / "methods" / "common.py").exists())
sys.path.insert(0, str(BENCHMARK))
AIRCHECK = BENCHMARK.parent
EXP_DIR = AIRCHECK / "output" / EXPERIMENT
print(f"Benchmark code : {BENCHMARK}")
print(f"Experiment     : {EXP_DIR}")

In [ ]:
FOLDS = sorted(EXP_DIR.glob("folds/*_cv_folds.npz"))
assert FOLDS, f"No folds in {EXP_DIR / 'folds'}; run 01_folds.ipynb first (or run_all.sh)"
FOLDS = FOLDS[0]
OUT = EXP_DIR / RUN_NAME
print(f"Folds : {FOLDS.name}")
print(f"Run   : {OUT}")

## Train and score

Everything happens in this one call, which writes `predictions.csv`, `run.json`, `progress.log` and `models/`
into the run folder. Follow a long run from a terminal with `tail -f <run folder>/progress.log`.

In [ ]:
from run import run_one

table = run_one("finetune", folds=FOLDS, test=TEST_PATH, out=OUT, num_workers=NUM_WORKERS,
                settings=dict(view=VIEW, epochs=EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE,
                              learning_rate=LEARNING_RATE, balance_classes=BALANCE_CLASSES, final=FINAL))
table.round(4)

The scores above come from aircheck-benchmark-core, the same code that scores every other model.
`cv_mean` and `cv_std` are over the folds.

## Every fold and epoch

In [ ]:
import json

per_epoch = pd.read_csv(OUT / "models" / "cv_metrics_per_epoch.csv")
selection = json.loads((OUT / "models" / "selection.json").read_text())
print(f"selected epoch {selection['best_epoch']} of {selection['epochs_trained']} trained, "
      f"by {selection['select_metrics']}")
per_epoch.round(4)

## The predictions and the models

In [ ]:
predictions = pd.read_csv(OUT / "predictions.csv")
print(predictions.groupby("split").size().to_string())
for path in sorted(OUT.rglob("best_model.ckpt")):
    print(f"{path.relative_to(OUT)}  {path.stat().st_size / 1e6:.0f} MB")
predictions.sort_values("score", ascending=False).head(5)

Score any molecules with these models in [04_inference.ipynb](04_inference.ipynb), and put every run
of the experiment in one report with [05_report.ipynb](05_report.ipynb).